# CommuteLive ETA evaluation
**Simulated data only.** This fictional route is separate from the unverified real pilot.
The experiment generates seeded simulator GPS and `arrivals.csv`, then replays GPS chronologically through the production PHP ingestion, traffic and arrivals services in disposable in-memory SQLite. No server, driver tokens, local database or prior logs are needed.
Polling predictions are evaluated against independent exact simulator ground truth, rather than GPS-derived arrival estimates. Both methods use identical paired samples. Predictions without an observed future arrival are reported as censored. Results describe this synthetic scenario and seed, not real ETA accuracy.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
if not (root / "backend").is_dir():
    root = root.parent
assert (root / "backend/vendor/autoload.php").exists(), "Run npm run setup first"
sys.path.insert(0, str(root / "analysis"))
from evaluate import evaluate, print_table
report = evaluate(root / "analysis/output/notebook", seed=42)
print_table(report)

In [ ]:
# Compare identical paired prediction sets; a negative delta means improvement.
for scenario in ("normal", "rush_hour", "incident", "sparse"):
    rows = {r["method"]: r for r in report["results"] if r["scenario"] == scenario}
    delta = rows["segment_blend_v1"]["mae_seconds"] - rows["naive_fixed_speed"]["mae_seconds"]
    print(f"{scenario}: MAE change versus fixed speed = {delta:+.2f} seconds")
assert report["gates"]["incident_red_within_120_seconds"]
assert report["gates"]["normal_has_countdown"]

## Interpretation and limits
Cold-start baselines are mode assumptions; no future history is used. Segment speeds include stopped pings, so dwells can influence both live speeds and the explicit intermediate-stop allowance. Weighting is by ping, not independent vehicle. Confidence labels are heuristics, not calibrated probabilities.
The naive method uses the configured fixed average speed (20 kph by default), without dwell. ETA polling is every 15 simulated seconds; speed aggregation every 30 seconds over a trailing five-minute window. Samples from the same trip are correlated; N counts displayed predictions, not independent trips. Repeat with additional seeds and verified routes before making accuracy claims. GPS arrival matching is approximate and is not substituted for exact simulator truth in the metrics.
Full input snapshots, flags, samples, ground truth, predictions, traffic states and result CSV/JSON are saved under `analysis/output/notebook/`. No output is committed. Real pilot geometry validation remains open.